# Datamine ADDMOD Process Example

This notebook demonstrates how to configure and run the **`addmod`** process wrapper in `dmstudio`.

## Process Description

## ADDMOD

Adds two models together by superimposing one onto another.

The resulting output model will contain all fields from both input models. Both input models must have the same origin, the same cell dimensions and the same number of cells in the three directions. However the models may have different numbers of sub-cells in equivalent cells, and the models may also have different cells undefined.

**ADDMOD** would be used for example to superimpose a surface model onto a grade model. This would enable the grade of all cells or sub-cells lying between two surfaces to be identified. If the surface model contains a rock type field showing which cells and sub-cells lie between the two surfaces and the grade model contains an interpolated grade field then the output model will contain both these fields.

If neither model contains any sub cells then the resulting output model will contain all cells which are in the two models. If a cell exists in one model but not in the other then default values will be assigned for those cells in the latter model. The resulting output model will be the same as if the two models had been added together using the **[JOIN](<join.md>)** process on the **IJK** field.

If one of the models, model A, contains sub-cells but the other one, model B, does not, then the output model will contain the same sub-cells as model A and each sub-cell will be assigned the field values from the appropriate cell of model B. If both models contain sub-cells but the sub-cells in a particular cell in one model are not t he same as the sub-cells in the same cell in the other model then the two sets are superimposed thus creating additional sub-cells in the output model.

Both input models must be sorted on the **IJK** field. Note that if the model editor was used to insert or split cells then the file will not be in the required order, and must be sorted first.

The @**TOLERNCE** parameter is used to define the smallest sub cell that will be written to the output model file. It is defined as a factor of the parent cell size in the three dimensions. If a subcell has dimensions xs, ys, zs and the parent cell has dimensions xp, yp, zp, then if xs<@**TOLERNCE** *xp or ys<@**TOLERNCE** *yp or zs<@**TOLERNCE** *zp then the sub cell is considered too small and will not be written to the output model. Care should be taken if the model is a seam type model with just a single cell in one direction. In such a case it may be necessary to set a value which is smaller than the default of 0.001 to avoid sub cells being eliminated unnecessarily.

**Note** : If a particular cell has different sets of sub-cells in the two input models then the superimposition of the sub-cells will create an output model with more sub-cells than either of the two input models. The data for each sub-cell is held as a single record in the model file. Therefore, if there is a significant disparity between sub-cells in the two input models then the output model will be much larger than the input models. Both input models must be sorted on the IJK field, for this process.

### Input Files:

* **in1** (Block Model):
  Model to be updated (sorted on IJK). Must contain at least the fields XC, YC, ZC, XINC,

## YINC, ZINC, XMORIG, YMORIG, ZMORIG, NX, NY, NZ, IJK.

  Required=Yes

* **in2** (Block Model):
  Update model (sorted on IJK).
  Required=Yes

### Output Files:

* **out** (Block Model):
  Output model.
  Required=Yes

### Fields:

### Parameters:

* **tolernce**:
  Defines the smallest cell that will be included in OUT. Defined as a factor of XINC,
  YINC, ZINC. Default = (0.001).
  Range=0,1
  Values=Undefined
  Default=0.001
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('addmod')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute addmod
print("Running addmod...")
dm_cmd.addmod(
    inmods_i=['optional'],  # required
    out_o='t_addmod_out',  # required
    # tolernce_p=0.001,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("addmod execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_addmod_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")